In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git@874b262"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import os

os.environ["WANDB_MODE"] = "disabled"

In [ ]:
PROMPT_TEMPLATE = """### Instruction:
Correct this sentence. (just give me the corrected sentence, no explanations needed)

### Input:
{input}

### Response:
"""


In [ ]:
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/dataset_fusionado.csv')
print(f"Loaded {len(df)} examples")

In [ ]:
import pandas as pd

def prepare_weighted_dataset(df, error_percentages, sample_size=10000):
    df_clean = df.copy()

    df_clean['error_type'] = df_clean['error_type'].str.split(';')
    df_clean = df_clean.explode('error_type')
    df_clean['error_type'] = df_clean['error_type'].str.strip()
    df_clean = df_clean.dropna(subset=['text', 'corrected_text', 'error_type'])

    sampled_dfs = []
    total_collected = 0

    total_percentage = sum(error_percentages.values())
    normalized = {k: v / total_percentage for k, v in error_percentages.items()}

    for error_type, proportion in normalized.items():
        n_samples = int(sample_size * proportion)
        subset = df_clean[df_clean['error_type'] == error_type]

        if len(subset) == 0:
            continue

        n_samples = min(n_samples, len(subset))
        sampled = subset.sample(n=n_samples, random_state=42)
        sampled_dfs.append(sampled)
        total_collected += len(sampled)

    if total_collected < sample_size:
        remaining = sample_size - total_collected
        remaining_df = df_clean.drop(pd.concat(sampled_dfs).index, errors='ignore')
        if len(remaining_df) > 0:
            extra = remaining_df.sample(n=min(remaining, len(remaining_df)), random_state=42)
            sampled_dfs.append(extra)

    balanced_df = pd.concat(sampled_dfs, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Sampled {len(balanced_df)} examples (target: {sample_size})")
    print(f"Top 15 error types:\n{balanced_df['error_type'].value_counts().head(15)}")

    training_data = []
    for _, row in balanced_df.iterrows():
        training_data.append({
            "input": row['text'].strip(),
            "output": row['corrected_text'].strip()
        })

    real_distribution = balanced_df['error_type'].value_counts(normalize=True) * 100
    comparison = pd.DataFrame({
        'target_%': pd.Series(error_percentages) / sum(error_percentages.values()) * 100,
        'real_%': real_distribution
    }).fillna(0).sort_values('real_%', ascending=False)

    print("\nComparison (Top 20):")
    print(comparison.head(20).round(2))

    return training_data


error_percentages = {
    "M:ADJ": 0.0357, "M:ADV": 0.0, "M:CONJ": 0.0, "M:DET": 0.4635, "M:NOUN": 2.1747,
    "M:OTHER": 2.246, "M:PART": 0.0357, "M:PREP": 0.107, "M:PRON": 0.3209, "M:PUNCT": 4.0285,
    "M:VERB": 0.1426, "M:VERB:FORM": 1.9251, "M:VERB:TENSE": 0.0357, "R:ADJ": 0.6061,
    "R:ADJ:FORM": 0.0713, "R:ADV": 0.1426, "R:DET": 1.4973, "R:MORPH": 0.3209, "R:NOUN": 11.6578,
    "R:NOUN:INFL": 0.0, "R:NOUN:NUM": 0.2496, "R:NOUN:POSS": 0.0357, "R:ORTH": 40.8556,
    "R:OTHER": 11.3725, "R:PREP": 0.4635, "R:PRON": 0.2139, "R:PUNCT": 0.3922, "R:SPELL": 16.4349,
    "R:VERB": 0.7843, "R:VERB:FORM": 1.3191, "R:VERB:INFL": 0.1783, "R:VERB:SVA": 0.1783,
    "R:VERB:TENSE": 0.2139, "R:WO": 0.0357, "U:ADJ": 0.0, "U:ADV": 0.0, "U:CONJ": 0.0,
    "U:DET": 0.0357, "U:NOUN": 0.0713, "U:OTHER": 0.1426, "U:PART": 0.0, "U:PREP": 0.2496,
    "U:PRON": 0.4278, "U:PUNCT": 0.3565, "U:VERB": 0.0357, "U:VERB:FORM": 0.107,
    "U:VERB:TENSE": 0.0357
}

training_data = prepare_weighted_dataset(df, error_percentages, sample_size=10000)
print(f"\nTotal training examples: {len(training_data)}")


In [ ]:
print("\nLoading model")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


In [ ]:
def format_prompts(examples):
    texts = []
    for input_text, output_text in zip(examples["input"], examples["output"]):
        text = PROMPT_TEMPLATE.format(input=input_text, output=output_text)
        texts.append(text)
    return {"text": texts}

train_data, val_data = train_test_split(
    training_data,
    test_size=0.1,
    random_state=42
)

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

train_dataset = Dataset.from_list(train_data).map(format_prompts, batched=True)
val_dataset = Dataset.from_list(val_data).map(format_prompts, batched=True)


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = TrainingArguments(
      per_device_train_batch_size = 8,
      gradient_accumulation_steps = 2,
      warmup_steps = 50,
      num_train_epochs = 2,
      learning_rate = 2e-6,
      fp16 = not torch.cuda.is_bf16_supported(),
      bf16 = torch.cuda.is_bf16_supported(),
      logging_steps = 50,
      eval_strategy = "steps",
      eval_steps = 200,
      save_steps = 400,
      save_total_limit = 2,
      load_best_model_at_end = True,
      optim = "adamw_8bit",
      weight_decay = 0.01,
      lr_scheduler_type = "linear",
      seed = 3407,
      output_dir = "outputs",
  ),

)

In [ ]:
print("Starting training")

trainer.train()

print("Training complete!")

In [ ]:
print("Saving model")
model.save_pretrained("grammar_checker_lora")
tokenizer.save_pretrained("grammar_checker_lora")

import shutil
shutil.copytree("grammar_checker_lora", "/content/drive/MyDrive/grammar_checker_lora", dirs_exist_ok=True)
print("Model saved to Google Drive at: /content/drive/MyDrive/grammar_checker_lora")


In [ ]:
FastLanguageModel.for_inference(model)

references_dict = {
    "I am second year undergraduate student from Mumbai..And I really want to learn fluent English..":
        "I am a second-year undergraduate student from Mumbai, and I really want to learn fluent English.",
    "my name joel i speak english and spanish, like to play with friends":
        "My name is Joel. I speak English and Spanish and I like to play with friends.",
    "she go to school everyday and she dont like homework":
        "She goes to school every day and she doesn't like homework.",
    "They was very happy yesterday..because they win the game":
        "They were very happy yesterday because they won the game.",
    "me and my friend goes to the park every day we plays football":
        "My friend and I go to the park every day and play football."
}

predictions = []
references = []

test_cases = [
    {
        "sentence": "I am second year undergraduate student from Mumbai..And I really want to learn fluent English..",
    },
    {
        "sentence": "my name joel i speak english and spanish, like to play with friends",
    },
    {
        "sentence": "she go to school everyday and she dont like homework",
    },
    {
        "sentence": "They was very happy yesterday..because they win the game",
    },
    {
        "sentence": "me and my friend goes to the park every day we plays football",
    }
]

print("TESTING MODEL")

for i, test in enumerate(test_cases, 1):
    test_sentence = test["sentence"]

    prompt = PROMPT_TEMPLATE.format(input=test_sentence)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        repetition_penalty=1.15,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    corrected = result.split('### Response:')[-1].strip()

    predictions.append(corrected)
    references.append(references_dict[test_sentence])


    print(f"Test {i}")
    print(f"Original:  {test_sentence}")
    print(f"Corrected: {corrected} \n")

print("All tests complete")


In [ ]:
%%capture
!pip install evaluate
!pip install rouge_score

In [ ]:
import evaluate
from nltk.translate.bleu_score import corpus_bleu

rouge = evaluate.load("rouge")
rouge_result = rouge.compute(predictions=predictions, references=references)

metric_descriptions = {
    "rouge1": "Overlap of individual words",
    "rouge2": "Overlap of word pairs",
    "rougeL": "Longest common subsequence",
    "rougeLsum": "Sentence-level structure overlap"
}

print("\nResults ROUGE:")
for k, v in rouge_result.items():
    desc = metric_descriptions.get(k, "")
    print(f"{k}: {v:.4f} — {desc}")

ref_bleu = [[r.split()] for r in references]
pred_bleu = [p.split() for p in predictions]
bleu_score = corpus_bleu(ref_bleu, pred_bleu)

print(f"\nBLEU Score: {bleu_score:.4f} — Precision of n-gram overlaps (literal similarity)")

for i, (inp, pred, ref) in enumerate(zip([c["sentence"] for c in test_cases], predictions, references)):
    print(f"\nTest {i+1}")
    print(f"Input:      {inp}")
    print(f"Prediction: {pred}")
    print(f"Reference:  {ref}")


In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import evaluate
from nltk.translate.bleu_score import corpus_bleu


dataset_path = "/content/drive/MyDrive/dataset_corrected.csv"
output_path = "/content/drive/MyDrive/results_with_predictions.csv"

df = pd.read_csv(dataset_path)

df_subset = df.head(200).copy()

texts = df_subset["text"].tolist()
references = df_subset["corrected_text"].tolist()


predictions = []
model.eval()

for sentence in tqdm(texts, desc="Generating predictions"):
    prompt = PROMPT_TEMPLATE.format(input=sentence)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            no_repeat_ngram_size=3
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    corrected = result.split('### Response:')[-1].strip()
    predictions.append(corrected)


df_subset["predicted_text"] = predictions
df_subset.to_csv(output_path, index=False)
print(f"\nPredictions saved to: {output_path}")


rouge = evaluate.load("rouge")
rouge_result = rouge.compute(predictions=predictions, references=references)

metric_descriptions = {
    "rouge1": "Overlap of individual words (word-level recall)",
    "rouge2": "Overlap of word pairs (measures fluency/coherence)",
    "rougeL": "Longest common subsequence (sentence structure)",
    "rougeLsum": "Sentence-level structure overlap (for multi-sentence summaries)"
}

print("\nResults ROUGE:")
for k, v in rouge_result.items():
    desc = metric_descriptions.get(k, "")
    print(f"{k}: {v:.4f} — {desc}")

ref_bleu = [[r.split()] for r in references]
pred_bleu = [p.split() for p in predictions]
bleu_score = corpus_bleu(ref_bleu, pred_bleu)

print(f"\nBLEU Score: {bleu_score:.4f} — Precision of n-gram overlaps (literal similarity)")


for i in range(5):
    print(f"\nExample {i+1}")
    print(f"Input:      {texts[i]}")
    print(f"Prediction: {predictions[i]}")
    print(f"Reference:  {references[i]}")

In [ ]:
"""Test Original Gemma Model Without Fine-tuning"""

from google.colab import drive
import torch
from unsloth import FastLanguageModel
import os

os.environ["WANDB_MODE"] = "disabled"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

test_cases = [
    {
        "sentence": "I am second year undergraduate student from Mumbai..And I really want to learn fluent English..",
    },
    {
        "sentence": "my name joel i speak english and spanish, like to play with friends",
    },
    {
        "sentence": "she go to school everyday and she dont like homework",
    },
    {
        "sentence": "They was very happy yesterday..because they win the game",
    },
    {
        "sentence": "me and my friend goes to the park every day we plays football",
    }
]

print("TESTING ORIGINAL MODEL (NO FINE-TUNING)")

for i, test in enumerate(test_cases, 1):
    test_sentence = test["sentence"]

    prompt = f"""### Instruction:
Correct this sentence. (just give me the corrected sentence, no explanations needed)

### Input:
{test_sentence}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        repetition_penalty=1.15,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    corrected = result.split('### Response:')[-1].strip()

    print(f"Test {i}")
    print(f"Original:  {test_sentence}")
    print(f"Corrected: {corrected}")
    print("-" * 50 + "\n")

print("All tests complete!")